# Dupire Local Volatility

## The problem local vol solves

BSM assumes a single constant $\sigma$. But the market smile shows implied vol varies
by strike and maturity - the market's terminal distribution of $S_T$ is *not* lognormal
(skewed, fat left tail: equity crash fear). A constant-$\sigma$ model produces a *flat*
smile and cannot fit this.

**Local volatility** replaces the constant $\sigma$ with a deterministic *function*
$\sigma_{\text{loc}}(S, t)$ - volatility that depends on spot and time, but adds no new
source of randomness. The risk-neutral SDE becomes:

$$dS_t = (r - q)\,S_t\,dt + \sigma_{\text{loc}}(S_t, t)\,S_t\,dW_t$$

**Dupire's theorem**: given the full surface of observed call prices $C(K, T)$, there is
a *unique* local-vol function reproducing every price simultaneously, recoverable by
differentiating the surface.

## What this is, and is NOT

- **Still** one-factor (a single $W_t$), arbitrage-free, and complete - so the entire
  risk-neutral pricing framework carries over.
- **No longer** geometric Brownian motion: with $\sigma_{\text{loc}}(S,t)$ varying, the
  log-SDE has non-constant coefficients, so $S_T$ is **not lognormal** and there is **no
  closed-form price** in general.
- BSM is the special case $\sigma_{\text{loc}} = \text{const}$. Local vol is the strictly
  larger model that contains it - and abandoning the constant-$\sigma$/lognormal
  straitjacket is *precisely* what lets it fit the smile.

The non-lognormality is the point, not a flaw: it is the only way to match a market whose
implied $S_T$ distribution is non-lognormal.

## Derivation roadmap

Three pieces combine to give Dupire's formula:
1. **Fokker-Planck** - the PDE for how the risk-neutral density evolves forward in time.
2. **Breeden-Litzenberger** - links that (unobservable) density to (observable) call prices.
3. **Substitute and solve** for $\sigma_{\text{loc}}^2$.

## 1. Fokker-Planck (forward Kolmogorov equation)

A diffusion process can be described by two dual PDEs for its transition density
$p(x_0, t_0; x, t)$ = the probability of being at $x$ at time $t$ given a start at
$(x_0, t_0)$:

- **Backward Kolmogorov** differentiates the *initial* variables $(x_0, t_0)$: fix where
  you observe, ask how the density depends on where you started. Pricing PDEs (like BSM)
  are backward equations - they propagate a fixed terminal payoff *back* to today.
- **Forward Kolmogorov = Fokker-Planck** differentiates the *terminal* variables $(x, t)$:
  fix the (known) starting point, watch the density evolve *forward* in time.## 2. Breeden-Litzenberger: density from prices

The density $p$ in Fokker-Planck is unobservable. This result expresses it in terms of
observable call prices. Start from the discounted-expected-payoff definition:

$$C(K, T) = e^{-rT}\int_K^\infty (S - K)\,p(S_0, t_0; S, T)\,dS$$

Differentiate once in $K$ (Leibniz rule - the $(S-K)$ term and the lower limit both
depend on $K$; the boundary term vanishes since the integrand is zero at $S=K$):

$$\frac{\partial C}{\partial K} = -e^{-rT}\int_K^\infty p\,dS$$

Differentiate again:

$$\boxed{\;\frac{\partial^2 C}{\partial K^2} = e^{-rT}\,p(S_0, t_0; K, T)\;}$$

**The second strike-derivative of the call price IS the discounted risk-neutral density**,
evaluated at $S = K$. This is the bridge: it lets us replace $p$ in Fokker-Planck with
$\partial^2 C / \partial K^2$, turning a statement about densities into a formula in
prices we can measure.

For a diffusion $dX = \mu(X,t)\,dt + \sigma(X,t)\,dW$, Fokker-Planck reads:

$$\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x}\big[\mu(x,t)\,p\big] + \frac12\frac{\partial^2}{\partial x^2}\big[\sigma^2(x,t)\,p\big]$$

Two terms with a clear physical reading:
- **Drift / advection** $-\partial_x[\mu p]$: the density's centre of mass moving.
- **Diffusion** $\tfrac12\partial_{xx}[\sigma^2 p]$: the density *spreading*, governed by
  $\sigma^2$. **This is the only term carrying $\sigma^2$** - the quantity we want to extract.

**Why Dupire needs the forward equation specifically**: it is parameterised by the
*terminal* variables. For options, the terminal variables are the **strike $K$ and
maturity $T$** - exactly the axes of the price surface we observe. The forward equation
lets us sweep the whole $(K, T)$ surface from a single starting point (today's spot),
which the backward equation (good for one option at a time) cannot.

## 2. Breeden-Litzenberger: density from prices

The density $p$ in Fokker-Planck is unobservable. This result expresses it in terms of
observable call prices. Start from the discounted-expected-payoff definition:

$$C(K, T) = e^{-rT}\int_K^\infty (S - K)\,p(S_0, t_0; S, T)\,dS$$

Differentiate once in $K$ (Leibniz rule - the $(S-K)$ term and the lower limit both
depend on $K$; the boundary term vanishes since the integrand is zero at $S=K$):

$$\frac{\partial C}{\partial K} = -e^{-rT}\int_K^\infty p\,dS$$

Differentiate again:

$$\boxed{\;\frac{\partial^2 C}{\partial K^2} = e^{-rT}\,p(S_0, t_0; K, T)\;}$$

**The second strike-derivative of the call price IS the discounted risk-neutral density**,
evaluated at $S = K$. This is the bridge: it lets us replace $p$ in Fokker-Planck with
$\partial^2 C / \partial K^2$, turning a statement about densities into a formula in
prices we can measure.

## 3. Dupire's formula

Substituting Breeden-Litzenberger into the forward equation for $C(K,T)$ and solving for
the diffusion coefficient gives:

$$\sigma_{\text{loc}}^2(K, T) = \frac{\dfrac{\partial C}{\partial T} + (r - q)\,K\,\dfrac{\partial C}{\partial K} + q\,C}{\tfrac12\,K^2\,\dfrac{\partial^2 C}{\partial K^2}}$$

**How to read it structurally** (so it is understood, not memorised):

- **Denominator** $\tfrac12 K^2 \partial^2 C/\partial K^2$ is *literally the diffusion term
  of Fokker-Planck*, with the density rewritten as $\partial^2 C/\partial K^2$. Since
  $\sigma^2$ *multiplied* the density term in the PDE, isolating it means *dividing* by
  that term, hence the density lands in the denominator.
- **Numerator** collects the time-evolution and drift terms: $\partial C/\partial T$ is
  the forward time-derivative ($\partial p/\partial t$), and $(r-q)K\,\partial C/\partial K + qC$
  are the drift pieces.

## The practical curse (why this is hard, not just elegant)

The density sits in the **denominator**, and $\partial^2 C/\partial K^2$ *is* that density
(Breeden-Litzenberger). In the far wings (deep OTM), the density is tiny - the stock is
very unlikely to finish there - so $\partial^2 C/\partial K^2 \to 0$. Dividing by a
near-zero number means:

> A small error in the second derivative of a *noisy* price surface, divided by a
> near-zero denominator, produces a wildly unstable $\sigma_{\text{loc}}$.

The formula is **exact in theory, a minefield in practice**. This is why local-vol
extraction demands a **smooth, arbitrage-free price/IV surface** *before* differentiating -
raw market quotes are far too noisy. Building that surface (interpolation, arbitrage
constraints, SVI or similar) is the real engineering work of local vol, and the reason
the denominator structure matters: it tells you exactly where the model will blow up.

## Limitation (motivates Heston, Week 4)

Local vol makes $\sigma$ a *deterministic* function of $(S, t)$ - volatility is fully
determined by where the stock is, with no independent randomness. It fits *today's* smile
exactly, but gets the smile's *dynamics* wrong (the smile rolls the wrong way as spot
moves). Stochastic-vol models (Heston) give volatility its own Brownian motion to fix this.

In [1]:
from models.bsm import bsm_price
from calibration.dupire import dupire_local_vol
import numpy as np

S0, r, q, sigma_true = 100.0, 0.05, 0.0, 0.20
K_grid = np.linspace(80, 120, 41)      # uniform strikes
T_grid = np.linspace(0.1, 2.0, 40)     # uniform maturities

# build C(K,T) grid from closed-form BSM (constant vol)
C = np.array([[bsm_price(S0, K, T, r, sigma_true, 'call') for K in K_grid]
              for T in T_grid])         # shape (n_T, n_K)

sigma_loc = dupire_local_vol(C, K_grid, T_grid, r, q)

print(f"mean {sigma_loc.mean():.4f}, std {sigma_loc.std():.4f}, "
      f"min {sigma_loc.min():.4f}, max {sigma_loc.max():.4f}")

mean 0.2055, std 0.0180, min 0.1905, max 0.3392


In [2]:
# where are the bad points? print the surface's deviation from 0.20
deviation = np.abs(sigma_loc - 0.20)
bad_T, bad_K = np.unravel_index(np.argmax(deviation), deviation.shape)
print(f"worst point: T={T_grid[bad_T]:.3f}, K={K_grid[bad_K]:.1f}, "
      f"sigma_loc={sigma_loc[bad_T, bad_K]:.4f}")

# and check: does trimming the wings clean it up?
interior = sigma_loc[1:-1, 5:-5]   # drop edge maturities and far strikes
print(f"interior only: mean {interior.mean():.4f}, std {interior.std():.4f}, "
      f"min {interior.min():.4f}, max {interior.max():.4f}")

worst point: T=0.100, K=120.0, sigma_loc=0.3392
interior only: mean 0.2000, std 0.0005, min 0.1958, max 0.2020
